# Formula 1 Driver Performance Index (5 Seasons)

## Project aim
Build a composite **F1 Driver Performance Index** that compares drivers across the **last 5 completed seasons**.

## Notebook structure
1. Setup and imports
2. Download race and qualifying data for 5 seasons
3. Build clean event-level datasets
4. Aggregate to driver-season level
5. Handle missing data
6. Multivariate analysis
7. Normalisation
8. Weighting and aggregation
9. Compare with official standings
10. Visualisation
11. Export final outputs
12. Reflection, struggles, references, and commit log


In [1]:
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import time

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

## 1. Data Setup

In [2]:
BASE_URL = 'https://api.jolpi.ca/ergast/f1'
SEASONS = [2021, 2022, 2023, 2024, 2025]

DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
OUTPUT_DIR = Path('outputs')

for directory in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEASONS

[2021, 2022, 2023, 2024, 2025]

## 2. Download data

In [3]:
def fetch_json(endpoint):
    url = f"{BASE_URL}/{endpoint}"
    
    response = requests.get(url, timeout=30)
    
    if response.status_code == 429:
        print("Rate limited. Waiting 2 seconds...")
        time.sleep(2)
        response = requests.get(url, timeout=30)
    
    response.raise_for_status()
    
    # small delay to avoid hitting limit
    time.sleep(0.3)
    
    return response.json()


def get_rounds_for_season(season: int) -> list[int]:
    data = fetch_json(f'{season}.json')
    races = data['MRData']['RaceTable']['Races']
    return [int(r['round']) for r in races]

In [5]:

def build_race_rows(data: dict[str, Any]) -> list[dict[str, Any]]:
    races = data['MRData']['RaceTable']['Races']
    rows = []

    for race in races:
        for result in race.get('Results', []):
            driver = result['Driver']
            constructor = result['Constructor']

            rows.append({
                'season': int(race['season']),
                'round': int(race['round']),
                'race_name': race['raceName'],
                'date': race.get('date'),
                'driver_id': driver['driverId'],
                'driver_code': driver.get('code', ''),
                'driver_name': f"{driver['givenName']} {driver['familyName']}",
                'constructor': constructor['name'],
                'grid': pd.to_numeric(result.get('grid'), errors='coerce'),
                'finish_position': pd.to_numeric(result.get('position'), errors='coerce'),
                'points': pd.to_numeric(result.get('points'), errors='coerce'),
                'status': result.get('status', '')
            })

    return rows


def build_quali_rows(data: dict[str, Any]) -> list[dict[str, Any]]:
    races = data['MRData']['RaceTable']['Races']
    rows = []

    for race in races:
        for result in race.get('QualifyingResults', []):
            driver = result['Driver']
            constructor = result['Constructor']

            rows.append({
                'season': int(race['season']),
                'round': int(race['round']),
                'race_name': race['raceName'],
                'date': race.get('date'),
                'driver_id': driver['driverId'],
                'driver_code': driver.get('code', ''),
                'driver_name': f"{driver['givenName']} {driver['familyName']}",
                'constructor': constructor['name'],
                'quali_position': pd.to_numeric(result.get('position'), errors='coerce'),
                'q1': result.get('Q1'),
                'q2': result.get('Q2'),
                'q3': result.get('Q3')
            })

    return rows

In [7]:
all_race_rows = []
all_quali_rows = []

for season in SEASONS:
    rounds = get_rounds_for_season(season)
    print(f"Season {season}: {len(rounds)} rounds found")

    for rnd in rounds:
        race_json = fetch_json(f"{season}/{rnd}/results.json")
        quali_json = fetch_json(f"{season}/{rnd}/qualifying.json")

        all_race_rows.extend(build_race_rows(race_json))
        all_quali_rows.extend(build_quali_rows(quali_json))

race_df = pd.DataFrame(all_race_rows)
quali_df = pd.DataFrame(all_quali_rows)

race_df.to_csv(RAW_DIR / "race_results_2021_2025.csv", index=False)
quali_df.to_csv(RAW_DIR / "qualifying_results_2021_2025.csv", index=False)

print("\nRace dataset shape:", race_df.shape)
print("Qualifying dataset shape:", quali_df.shape)

print("\nRace data preview:")
display(race_df.head())

print("\nQualifying data preview:")
display(quali_df.head())

Season 2021: 22 rounds found
Season 2022: 22 rounds found
Season 2023: 22 rounds found
Season 2024: 24 rounds found
Season 2025: 24 rounds found

Race dataset shape: (2278, 12)
Qualifying dataset shape: (2277, 12)

Race data preview:


,season,round,race_name,date,driver_id,driver_code,driver_name,constructor,grid,finish_position,points,status
0,2021,1,Bahrain Grand Prix,2021-03-28,hamilton,HAM,Lewis Hamilton,Mercedes,2,1,25.0,Finished
1,2021,1,Bahrain Grand Prix,2021-03-28,max_verstappen,VER,Max Verstappen,Red Bull,1,2,18.0,Finished
2,2021,1,Bahrain Grand Prix,2021-03-28,bottas,BOT,Valtteri Bottas,Mercedes,3,3,16.0,Finished
3,2021,1,Bahrain Grand Prix,2021-03-28,norris,NOR,Lando Norris,McLaren,7,4,12.0,Finished
4,2021,1,Bahrain Grand Prix,2021-03-28,perez,PER,Sergio Pérez,Red Bull,0,5,10.0,Finished



Qualifying data preview:


,season,round,race_name,date,driver_id,driver_code,driver_name,constructor,quali_position,q1,q2,q3
0,2021,1,Bahrain Grand Prix,2021-03-28,max_verstappen,VER,Max Verstappen,Red Bull,1,1:30.499,1:30.318,1:28.997
1,2021,1,Bahrain Grand Prix,2021-03-28,hamilton,HAM,Lewis Hamilton,Mercedes,2,1:30.617,1:30.085,1:29.385
2,2021,1,Bahrain Grand Prix,2021-03-28,bottas,BOT,Valtteri Bottas,Mercedes,3,1:31.200,1:30.186,1:29.586
3,2021,1,Bahrain Grand Prix,2021-03-28,leclerc,LEC,Charles Leclerc,Ferrari,4,1:30.691,1:30.010,1:29.678
4,2021,1,Bahrain Grand Prix,2021-03-28,gasly,GAS,Pierre Gasly,AlphaTauri,5,1:30.848,1:30.513,1:29.809


## 3. EVENT-LEVEL FEATURE ENGINEERING

In [ ]:
race_df["is_win"] = (race_df["finish_position"] == 1).astype(int)

race_df["is_podium"] = race_df["finish_position"].isin([1, 2, 3]).astype(int)

race_df["is_top10"] = race_df["finish_position"].between(1, 10, inclusive="both").fillna(False).astype(int)

race_df["is_dnf"] = (~race_df["status"].str.contains("Finished", case=False, na=False)).astype(int)

race_df["positions_gained"] = race_df["grid"] - race_df["finish_position"]

race_df.head()

,season,round,race_name,date,driver_id,driver_code,driver_name,constructor,grid,finish_position,points,status,is_win,is_podium,is_top10,is_dnf,positions_gained
0,2021,1,Bahrain Grand Prix,2021-03-28,hamilton,HAM,Lewis Hamilton,Mercedes,2,1,25.0,Finished,1,1,1,0,1
1,2021,1,Bahrain Grand Prix,2021-03-28,max_verstappen,VER,Max Verstappen,Red Bull,1,2,18.0,Finished,0,1,1,0,-1
2,2021,1,Bahrain Grand Prix,2021-03-28,bottas,BOT,Valtteri Bottas,Mercedes,3,3,16.0,Finished,0,1,1,0,0
3,2021,1,Bahrain Grand Prix,2021-03-28,norris,NOR,Lando Norris,McLaren,7,4,12.0,Finished,0,0,1,0,3
4,2021,1,Bahrain Grand Prix,2021-03-28,perez,PER,Sergio Pérez,Red Bull,0,5,10.0,Finished,0,0,1,0,-5
